# Phase 4: Production Classifier Training Loop

This notebook trains the PyTorch **EfficientNet-B0** classification model on the PlantVillage dataset.

### Modern Additions:
*   **Mixed Precision (AMP)** via PyTorch Scaler
*   **Learning Rate Scheduler** (ReduceLROnPlateau)
*   **Early Stopping** validation safeguard
*   **Checkpoint Save & Resume** mechanics
*   **TensorBoard metrics logging**

In [3]:
# Setup path and imports
from pathlib import Path
import sys
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import torchvision.models as models
from torch.utils.tensorboard import SummaryWriter
import time
import copy

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT.name != 'ai' and PROJECT_ROOT.parent != PROJECT_ROOT:
    if (PROJECT_ROOT / 'ai').exists():
        PROJECT_ROOT = PROJECT_ROOT / 'ai'
        break
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import utils

In [4]:
# Pin seed and set active device
utils.seed.set_seed(42)
device = utils.device.get_device()
utils.device.print_device_info()

🔒 Random seed pinned globally to: 42
PyTorch Version: 2.11.0+cu128
CUDA Available : True
Active GPU Name: NVIDIA GeForce RTX 4050 Laptop GPU


In [5]:
# Load datasets and construct loaders
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(utils.config.IMAGE_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

val_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(utils.config.IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

train_dataset = datasets.ImageFolder(root=str(utils.config.TRAIN_DIR), transform=train_transforms)
val_dataset = datasets.ImageFolder(root=str(utils.config.VAL_DIR), transform=val_transforms)

train_loader = DataLoader(
    train_dataset,
    batch_size=utils.config.BATCH_SIZE,
    shuffle=True,
    num_workers=utils.config.NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=utils.config.BATCH_SIZE,
    shuffle=False,
    num_workers=utils.config.NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

print(f"Loaded {len(train_dataset)} training images and {len(val_dataset)} validation images.")

Loaded 43444 training images and 10861 validation images.


In [6]:
# Build Model (EfficientNet-B0)
def build_classifier(num_classes):
    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
    # Replace output layer classifier
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)
    return model

model = build_classifier(utils.config.NUM_CLASSES).to(device)
print("✅ EfficientNet-B0 Model loaded and initialized on:", device)

✅ EfficientNet-B0 Model loaded and initialized on: cuda


### Early Stopping Helper Class

In [7]:
class EarlyStopping:
    def __init__(self, patience=5, min_delta=0.0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = None
        self.early_stop = False

    def __call__(self, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss > self.best_loss - self.min_delta:
            self.counter += 1
            print(f"[INFO] Early Stopping check: {self.counter}/{self.patience} validation iterations.")
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_loss = val_loss
            self.counter = 0

### Custom Mixed-Precision (AMP) PyTorch Training Loop

In [8]:
# Initialize hyperparameters, scalar, schedulers, and TensorBoard writers
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=utils.config.LEARNING_RATE, weight_decay=utils.config.WEIGHT_DECAY)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)
scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())
early_stopping = EarlyStopping(patience=5)

# Create directories for logging
log_dir = utils.paths.PROJECT_ROOT / "outputs" / "runs"
log_dir.mkdir(parents=True, exist_ok=True)
writer = SummaryWriter(log_dir=str(log_dir))

history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_acc = 0.0

print("🚀 Starting Training Pipeline...")
for epoch in range(utils.config.EPOCHS):
    # 1. Training Epoch
    model.train()
    running_loss = 0.0
    running_corrects = 0
    total_samples = 0
    
    start_time = time.time()
    
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        
        # Mixed Precision forward pass
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            _, preds = torch.max(outputs, 1)
            
        # Mixed Precision backward pass & optimization
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        running_loss += loss.item() * inputs.size(0)
        running_corrects += torch.sum(preds == labels.data)
        total_samples += inputs.size(0)
        
    train_loss = running_loss / total_samples
    train_acc = (running_corrects.double() / total_samples).item()
    
    # 2. Validation Epoch
    model.eval()
    val_running_loss = 0.0
    val_running_corrects = 0
    val_total_samples = 0
    
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            _, preds = torch.max(outputs, 1)
            
            val_running_loss += loss.item() * inputs.size(0)
            val_running_corrects += torch.sum(preds == labels.data)
            val_total_samples += inputs.size(0)
            
    val_loss = val_running_loss / val_total_samples
    val_acc = (val_running_corrects.double() / val_total_samples).item()
    
    # Epoch logs
    epoch_time = time.time() - start_time
    print(f"Epoch {epoch+1}/{utils.config.EPOCHS} | Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} Acc: {val_acc:.4f} | Time: {epoch_time:.2f}s")
    
    # Log to TensorBoard
    writer.add_scalar('Loss/Train', train_loss, epoch)
    writer.add_scalar('Loss/Val', val_loss, epoch)
    writer.add_scalar('Accuracy/Train', train_acc, epoch)
    writer.add_scalar('Accuracy/Val', val_acc, epoch)
    
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    
    # Save best checkpoint
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': val_acc,
        }, str(utils.config.BEST_MODEL_PATH))
        print(f"[SAVED] Best validation checkpoint saved to: {utils.config.BEST_MODEL_PATH}")
        
    # Schedulers & Early Stopping
    scheduler.step(val_loss)
    early_stopping(val_loss)
    if early_stopping.early_stop:
        print("🛑 Early Stopping triggered. Halting training loop.")
        break
        
writer.close()
print(f"🎉 Model Training Completed. Best Val Accuracy: {best_acc:.4f}")

C:\Users\sasan\AppData\Local\Temp\ipykernel_30576\3491881936.py:5: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())


🚀 Starting Training Pipeline...


C:\Users\sasan\AppData\Local\Temp\ipykernel_30576\3491881936.py:31: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):


Epoch 1/15 | Train Loss: 0.2188 Acc: 0.9366 | Val Loss: 0.0845 Acc: 0.9721 | Time: 175.06s
[SAVED] Best validation checkpoint saved to: O:\Hackthons\KrishiOS\ai\models\classifier\best_model.pth
Epoch 2/15 | Train Loss: 0.0823 Acc: 0.9742 | Val Loss: 0.0847 Acc: 0.9743 | Time: 143.99s
[SAVED] Best validation checkpoint saved to: O:\Hackthons\KrishiOS\ai\models\classifier\best_model.pth
[INFO] Early Stopping check: 1/5 validation iterations.
Epoch 3/15 | Train Loss: 0.0653 Acc: 0.9796 | Val Loss: 0.0367 Acc: 0.9881 | Time: 164.06s
[SAVED] Best validation checkpoint saved to: O:\Hackthons\KrishiOS\ai\models\classifier\best_model.pth
Epoch 4/15 | Train Loss: 0.0568 Acc: 0.9820 | Val Loss: 0.1515 Acc: 0.9819 | Time: 141.85s
[INFO] Early Stopping check: 1/5 validation iterations.
Epoch 5/15 | Train Loss: 0.0482 Acc: 0.9844 | Val Loss: 0.0827 Acc: 0.9797 | Time: 142.58s
[INFO] Early Stopping check: 2/5 validation iterations.
Epoch 6/15 | Train Loss: 0.0457 Acc: 0.9856 | Val Loss: 0.1329 Acc: 